In [ ]:
!nvidia-smi

Fri Apr 24 15:06:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers rank_bm25 faiss-cpu joblib evaluate rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
import evaluate

rouge = evaluate.load('rouge')
chrf = evaluate.load('chrf')

def calculate_metrics(predictions, references):
    rouge_results = rouge.compute(predictions=predictions, references=references)

    chrf_results = chrf.compute(predictions=predictions, references=references)

    return {
        "ROUGE-L": round(rouge_results['rougeL'], 4),
        "chrF++": round(chrf_results['score'], 4)
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import numpy as np

def calculate_recall(retrieved_indices, ground_truth_index, k=5):
    top_k_results = retrieved_indices[:k]

    return 1 if ground_truth_index in top_k_results else 0



In [ ]:
import joblib
import faiss
import json
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
path = "/content/drive/MyDrive/Studia_MJ/LLM_projekt"

data = joblib.load(f'{path}/data_pack.joblib')
documents = data['documents']
bm25 = data['bm25']

index = faiss.read_index(f"{path}/faiss_index.bin")

with open(f'{path}/synthetic_test_set.json', 'r', encoding='utf-8') as f:
    test_set = json.load(f)

print("Data is loaded")

Mounted at /content/drive
Data is loaded


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss


embed_model = SentenceTransformer("microsoft/harrier-oss-v1-0.6b")
embeddings = embed_model.encode(documents,
    batch_size=4,
    show_progress_bar=True,
    convert_to_numpy=True
)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"Index FAISS size: {index.ntotal}")

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

Index FAISS size: 1020


In [ ]:
def search_faiss(query, embed_model, index, k=5):
  query_vector = embed_model.encode([query]).astype('float32')
  distances, indices = index.search(query_vector, k)
  return indices[0].tolist()

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

rerank_model_id = 'BAAI/bge-reranker-v2-m3'
rerank_tokenizer = AutoTokenizer.from_pretrained(rerank_model_id)
rerank_model = AutoModelForSequenceClassification.from_pretrained(rerank_model_id)

rerank_model.eval()
rerank_model.to("cpu")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=1024, out_features=1, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(8194, 1024, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
             

In [ ]:
def run_reranker(query, candidate_docs, candidate_indices, top_k=3):
    pairs = [[query, doc] for doc in candidate_docs]

    with torch.no_grad():
        inputs = rerank_tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
        inputs = {k: v.to(rerank_model.device) for k, v in inputs.items()}

        scores = rerank_model(**inputs, return_dict=True).logits.view(-1, ).float()

        sorted_indices_in_batch = torch.argsort(scores, descending=True)

        final_indices = [candidate_indices[i] for i in sorted_indices_in_batch[:top_k]]
        return final_indices

In [ ]:
def generate_answer_variant_b(question, context, model, tokenizer):
    prompt_with_context = f"""
    Answer the user question based ONLY on the provided context.
    If the information is not in the context, say that you don't know.

    Context:
    {context}

    Question:
    {question}
    """

    messages = [
        {"role": "system", "content": "You are a factual cosmetic assistant. Use the provided context to answer questions."},
        {"role": "user", "content": prompt_with_context}
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
from tqdm import tqdm
results_variant_d = []
all_recalls_d = []


for item in tqdm(test_set[:40]):
    query = item['question']
    ref = item['expected_answer']
    gt_idx = item['document_i']


    initial_indices = search_faiss(query, embed_model, index, k=20)
    initial_docs = [documents[i] for i in initial_indices]

    final_indices = run_reranker(query, initial_docs, initial_indices, top_k=3)

    recall = 1 if gt_idx in final_indices else 0
    all_recalls_d.append(recall)

    context = "\n---\n".join([documents[i] for i in final_indices])
    pred = generate_answer_variant_b(query, context, model, tokenizer)
    print(f"Query: {query} Pred: {pred}")

    results_variant_d.append({"pred": pred, "ref": ref})

metrics_d = calculate_metrics([r['pred'] for r in results_variant_d], [r['ref'] for r in results_variant_d])

print(f"\n--- Variant D (Dense + Reranker) ---")
print(f"ROUGE-L:  {metrics_d['ROUGE-L']}")
print(f"chrF++:   {metrics_d['chrF++']}")
print(f"Recall@3: {np.mean(all_recalls_d)}")


  2%|▎         | 1/40 [02:00<1:18:20, 120.53s/it]

Query: What is the primary function of Methyl Gluceth-10 in cosmetic formulations? Pred: The primary function of Methyl Gluceth-10 is a humectant ingredient helping the skin to cling onto water.



  5%|▌         | 2/40 [03:45<1:10:35, 111.47s/it]

Query: What is the source of Leuconostoc/Radish Root Ferment Filtrate and what makes it a good preservative in cosmetics? Pred: The source of Leuconostoc/Radish Root Ferment Filtrate is radishes fermented with Leuconostoc kimchii, a lactic acid bacteria. It's a good preservative in cosmetics due to the antimicrobial properties of a peptide secreted from the bacteria during the fermentation process.



  8%|▊         | 3/40 [05:24<1:05:03, 105.51s/it]

...


 98%|█████████▊| 39/40 [56:49<01:31, 91.90s/it]

Query: What is the primary function of palmitic acid in skincare products, and in what types of products can it be found according to the document content? Pred: According to the context, the primary function of palmitic acid in skincare products is to make the skin feel nice and smooth, and it can be found in two types of products:

1. Moisturizers (emollients)
2. Cleansers (as a foam building cleansing agent)
3. Shaving foams



100%|██████████| 40/40 [58:18<00:00, 87.47s/it]

Query: What are the potential risks associated with Symphytum Officinale Root Extract in cosmetics and how are they regulated in Germany? Pred: The potential risk associated with Symphytum Officinale Root Extract in cosmetics is the presence of pyrrolizidine alkaloids (PA), which have proven toxic effects. In Germany, the permissible dose of PA is 100 μg/day, and the treatment is limited to 4-6 weeks per year if the dose is between 10-100 μg/day.



--- Variant D (Dense + Reranker) ---
ROUGE-L:  0.5134
chrF++:   62.5006
Recall@3: 0.975
